In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("lakehouse").getOrCreate()


spark.sql("DROP TABLE IF EXISTS nessie.silver.ohlcv")

DataFrame[]

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Grab your existing active notebook Spark Session
spark = SparkSession.builder.getOrCreate()

# 2. Path to two files directly from your bronze bucket
test_paths = [
    "s3a://lakehouse/bronze/ohlcv/1min/AAPL_raw_1min.parquet",
    "s3a://lakehouse/bronze/ohlcv/1min/ABNB_raw_1min.parquet"
]

print("--- Reading raw parquet test files ---")
# Explicitly selecting the hidden column '_metadata' alongside everything else
df_raw = spark.read.parquet(*test_paths).select("*", "_metadata")

# 3. Apply the bulletproof hidden metadata extraction regex
df_with_ticker = df_raw.withColumn(
    "ticker", 
    F.regexp_extract(F.col("_metadata.file_name"), r"([^/]+)_raw", 1)
)

print("\n--- [TEST 1] Checking Ticker Extraction Before Shuffle ---")
# Changed "date" to "datetime" to match your schema
df_with_ticker.select("datetime", "Open", "Close", "ticker").show(5)

# 4. Stress Test: Trigger a heavy window partition
# Changed "date" to "datetime" here as well
window_spec = Window.partitionBy("ticker", "datetime").orderBy(F.lit(1))
df_shuffled = df_with_ticker.withColumn("_row_num", F.row_number().over(window_spec)) \
                            .filter(F.col("_row_num") == 1) \
                            .drop("_row_num")

print("\n--- [TEST 2] Checking Ticker Columns After Shuffle Boundary ---")
df_shuffled.select("datetime", "Open", "Close", "ticker").show(5)

--- Reading raw parquet test files ---

--- [TEST 1] Checking Ticker Extraction Before Shuffle ---
+-------------------+----+-----+------+
|           datetime|Open|Close|ticker|
+-------------------+----+-----+------+
|1041240600000000000|0.23| 0.23|  AAPL|
|1041240660000000000|0.23| 0.23|  AAPL|
|1041240720000000000|0.23| 0.23|  AAPL|
|1041240780000000000|0.23| 0.23|  AAPL|
|1041240840000000000|0.23| 0.23|  AAPL|
+-------------------+----+-----+------+
only showing top 5 rows


--- [TEST 2] Checking Ticker Columns After Shuffle Boundary ---


[Stage 11:==================================================>     (10 + 1) / 11]

+-------------------+----+-----+------+
|           datetime|Open|Close|ticker|
+-------------------+----+-----+------+
|1041241140000000000|0.23| 0.23|  AAPL|
|1041241500000000000|0.23| 0.23|  AAPL|
|1041241740000000000|0.23| 0.23|  AAPL|
|1041242280000000000|0.23| 0.23|  AAPL|
|1041242340000000000|0.23| 0.23|  AAPL|
+-------------------+----+-----+------+
only showing top 5 rows



In [7]:
df  =spark.read.parquet("s3a://lakehouse/bronze/ohlcv/1min/AAPL_raw_1min.parquet")

In [8]:
df.show()


+-------------------+----+----+----+-----+-------+---------+
|           datetime|Open|High| Low|Close| Volume|   source|
+-------------------+----+----+----+-----+-------+---------+
|1041240600000000000|0.23|0.23|0.23| 0.23|2285864|pitrading|
|1041240660000000000|0.23|0.23|0.23| 0.23| 683200|pitrading|
|1041240720000000000|0.23|0.23|0.23| 0.23| 441280|pitrading|
|1041240780000000000|0.23|0.23|0.23| 0.23| 112000|pitrading|
|1041240840000000000|0.23|0.23|0.23| 0.23| 495600|pitrading|
|1041240900000000000|0.23|0.23|0.23| 0.23| 196000|pitrading|
|1041240960000000000|0.23|0.23|0.23| 0.23| 246456|pitrading|
|1041241020000000000|0.23|0.23|0.23| 0.23| 112000|pitrading|
|1041241080000000000|0.23|0.23|0.23| 0.23| 686952|pitrading|
|1041241140000000000|0.23|0.23|0.23| 0.23| 380800|pitrading|
|1041241200000000000|0.23|0.23|0.23| 0.23| 285600|pitrading|
|1041241260000000000|0.23|0.23|0.23| 0.23| 751520|pitrading|
|1041241320000000000|0.23|0.23|0.23| 0.23| 414400|pitrading|
|1041241380000000000|0.2

In [12]:
bronze = spark.read.option('header',True).option('inferSchema',True).parquet("s3a://lakehouse/bronze/ohlcv")

In [13]:
bronze.count()

921994682

In [11]:
bronze.show()

+-------------------+-----+-----+-----+-----+------+---------+----------+
|           datetime| Open| High|  Low|Close|Volume|   source|      date|
+-------------------+-----+-----+-----+-----+------+---------+----------+
|1041240600000000000|18.85|19.01|18.82| 19.0| 69211|pitrading|2026-06-11|
|1041240660000000000| 19.0|19.09|18.98|19.04| 53500|pitrading|2026-06-11|
|1041240720000000000|19.04|19.05|18.95|19.01| 61473|pitrading|2026-06-11|
|1041240780000000000|19.03|19.04|18.93|18.94| 80975|pitrading|2026-06-11|
|1041240840000000000|18.97|18.99|18.94|18.98| 28550|pitrading|2026-06-11|
|1041240900000000000|18.96|19.01|18.96|18.97| 67100|pitrading|2026-06-11|
|1041240960000000000|18.98|19.02|18.97| 19.0| 80512|pitrading|2026-06-11|
|1041241020000000000|19.03|19.05|18.99|19.02|140175|pitrading|2026-06-11|
|1041241080000000000|19.01|19.04|19.01|19.04| 22643|pitrading|2026-06-11|
|1041241140000000000|19.03|19.07|19.02|19.06| 39650|pitrading|2026-06-11|
|1041241200000000000|19.06|19.08|19.05

26/06/14 15:15:25 WARN NettyRpcEnv: Ignored failure: java.util.concurrent.TimeoutException: Cannot receive any reply from d932d49739f7:33381 in 10000 milliseconds
